# DeepGuard AI — Forensic Pipeline Demo
Runs `app/ml_features.py` on synthetic camera-like vs. generator-like images.

In [ ]:
import sys, os, io
sys.path.insert(0, os.path.abspath(".."))
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from app.ml_features import (
    analyze_image_bytes, noise_residual_score, _load_grayscale,
)
np.random.seed(0)
%matplotlib inline

In [ ]:
def camera_like_image(size=256, noise_std=8.0, seed=0):
    rng = np.random.default_rng(seed)
    base = np.zeros((size, size, 3), dtype=np.float64)
    for i in range(size):
        base[i, :, 0] = 90 + i * (60 / size)
        base[i, :, 1] = 70 + i * (50 / size)
        base[i, :, 2] = 110 + i * (40 / size)
    noisy = np.clip(base + rng.normal(0, noise_std, base.shape), 0, 255).astype("uint8")
    return Image.fromarray(noisy)

def generator_like_image(size=256):
    arr = np.full((size, size, 3), (150, 120, 170), dtype="uint8")
    t = size // 3
    arr[t:2*t, t:2*t] = (200, 190, 220)
    return Image.fromarray(arr)

def to_bytes(img):
    buf = io.BytesIO(); img.save(buf, format="PNG"); return buf.getvalue()

# Signal comparison + noise sensitivity sweep
img_camera, img_generator = camera_like_image(), generator_like_image()
result_camera = analyze_image_bytes(to_bytes(img_camera))
result_generator = analyze_image_bytes(to_bytes(img_generator))
print("Camera-like:", result_camera["fake_probability"], "% fake |", result_camera["signals"])
print("Generator-like:", result_generator["fake_probability"], "% fake |", result_generator["signals"])

noise_levels = [0, 8, 16, 24, 32]
residuals = [noise_residual_score(_load_grayscale(to_bytes(camera_like_image(noise_std=n)))[1]) for n in noise_levels]
plt.plot(noise_levels, residuals, marker="o")
plt.xlabel("Injected noise std-dev"); plt.ylabel("Noise-residual variance")
plt.title("Noise-residual score vs. noise level"); plt.grid(alpha=0.3); plt.show()

Classical signal-processing heuristics, not a trained classifier. See `tests/test_ml_features.py` and `scripts/evaluate_pipeline.py` for validation.